# 문제 2-2 : 영화 리뷰 감정 분석기

In [3]:
import os
from enum import Enum
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.exceptions import OutputParserException
from langchain_classic.output_parsers import EnumOutputParser, OutputFixingParser

load_dotenv()
PROVIDER = "groq"
CFG = {
    "groq": {"key": os.getenv("GROQ_API_KEY"), "url": "https://api.groq.com/openai/v1", "model": "openai/gpt-oss-120b"},
    "openai": {"key": os.getenv("OPENAI_API_KEY"), "url": None, "model": "gpt-4o-mini"},
}
c = CFG[PROVIDER]
if not c["key"]:
    raise RuntimeError(f".env 파일에 {PROVIDER.upper()}_API_KEY 가 없습니다.")

llm = ChatOpenAI(api_key=c["key"], base_url=c["url"], model=c["model"], temperature=0)


class Sentiment(Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"


parser = EnumOutputParser(enum=Sentiment)
fixing = OutputFixingParser.from_llm(llm=llm, parser=parser, max_retries=1)
print(parser.get_format_instructions())

pt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 리뷰 감정 분석 전문가입니다. 리뷰를 읽고 감정을 분류하세요. 설명 없이 값만 답하세요.\n{fmt}"),
    ("user", "리뷰: {review}"),
]).partial(fmt=parser.get_format_instructions())

chain = pt | llm | fixing

tests = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
]
u = input("영화 리뷰를 입력하세요 (엔터만 누르면 테스트 리뷰 3개 실행): ").strip()
if u:
    tests = [u]

for i, t in enumerate(tests, 1):
    try:
        r = chain.invoke({"review": t})
        print(f"[{i}] 리뷰: {t}")
        print(f"    감정: {r.value} ({r.name})")
    except OutputParserException as e:
        print(f"[{i}] 파싱 실패: {str(e).splitlines()[0]}")
    print("-" * 50)


Select one of the following options: 긍정, 부정, 보통
[1] 리뷰: 이 영화 정말 재미없어요. 시간 낭비였습니다.
    감정: 부정 (NEGATIVE)
--------------------------------------------------
[2] 리뷰: 배우들의 연기가 훌륭하고 스토리도 감동적이었어요!
    감정: 긍정 (POSITIVE)
--------------------------------------------------
[3] 리뷰: 그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.
    감정: 보통 (NEUTRAL)
--------------------------------------------------
